In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import textstat

df = pd.read_csv('../output/lada/legalbench_lada_2k.csv')

In [3]:
df.shape

(1997, 5)

In [7]:
from collections import defaultdict

def cap_by_type(df_in, type_col='type', max_per_type=25):
    # preserve the original row order while limiting each type to max_per_type
    counts = defaultdict(int)
    keep_idx = []
    for idx, t in zip(df_in.index, df_in[type_col]):
        if counts[t] < max_per_type:
            keep_idx.append(idx)
            counts[t] += 1
    return df_in.loc[keep_idx]

# F1 analysis

In [8]:
top_f1 = df.sort_values('Factor_1', ascending=False).head(100)

top_f1 = cap_by_type(top_f1)

print(top_f1['type'].value_counts())

top_f2 = df.sort_values('Factor_2', ascending=False).head(100)

top_f2 = cap_by_type(top_f2)

type
international_citizenship    25
Name: count, dtype: int64


In [9]:
word_pattern = re.compile(r"\b\w+\b")
sentiment_analyzer = SentimentIntensityAnalyzer()

def compute_text_metrics(series, include_sentiment_label=False):
    metrics = []
    for text in series.fillna(""):
        text_str = str(text)
        tokens = word_pattern.findall(text_str.lower())
        token_lengths = [len(token) for token in tokens]
        question_length = len(tokens)
        avg_length = float(np.mean(token_lengths)) if token_lengths else 0.0
        burstiness = float(np.std(token_lengths) / avg_length) if token_lengths and avg_length else 0.0
        if token_lengths:
            counts = Counter(tokens)
            total = sum(counts.values())
            probs = np.array(list(counts.values()), dtype=float) / total
            entropy = float(-np.sum(probs * np.log(probs)))
            perplexity = float(np.exp(entropy))
        else:
            perplexity = 0.0
        flesch_kincaid = float(textstat.flesch_kincaid_grade(text_str)) if text_str.strip() else 0.0
        sentiment_scores = sentiment_analyzer.polarity_scores(text_str) if text_str.strip() else {"compound": 0.0}
        compound_sentiment = float(sentiment_scores.get("compound", 0.0))
        record = {
            "question": text_str,
            "question_length": question_length,
            "average_word_length": avg_length,
            "burstiness": burstiness,
            "perplexity": perplexity,
            "flesch_kincaid_grade": flesch_kincaid,
            "sentiment_compound": compound_sentiment
        }
        if include_sentiment_label:
            if compound_sentiment >= 0.05:
                sentiment_label = "positive"
            elif compound_sentiment <= -0.05:
                sentiment_label = "negative"
            else:
                sentiment_label = "neutral"
            record["sentiment_label"] = sentiment_label
        metrics.append(record)
    return pd.DataFrame(metrics)

In [10]:
df_metrics = compute_text_metrics(df['question'])
top_f1_metrics = compute_text_metrics(top_f1['question'])
top_f2_metrics = compute_text_metrics(top_f2['question'])


summary_columns = ["question_length", "average_word_length", "burstiness", "perplexity", "flesch_kincaid_grade", "sentiment_compound"]
summary_df = pd.concat(
    [
        df_metrics[summary_columns].mean().rename("overall_mean"),
        top_f1_metrics[summary_columns].mean().rename("top_f1_mean"),
        top_f2_metrics[summary_columns].mean().rename("top_f2_mean")
    ],
    axis=1
)

display(summary_df)

for label, metrics_df in [
    ("overall", df_metrics),
    ("top_f1", top_f1_metrics),
    ("top_f2", top_f2_metrics)
]:
    print(f"\nSample metrics for {label} questions:")
    display(metrics_df.head(5))
    if "sentiment_label" in metrics_df.columns:
        print("Sentiment label distribution:")
        display(metrics_df['sentiment_label'].value_counts(normalize=True).rename(lambda x: f"{x} ({label})"))

,overall_mean,top_f1_mean,top_f2_mean
question_length,256.478217,42.560000,100.333333
average_word_length,5.226676,5.139671,5.292403
burstiness,0.568251,0.608640,0.581392
perplexity,72.579784,28.980972,40.192591
flesch_kincaid_grade,13.632242,13.938910,13.420673
sentiment_compound,0.207861,-0.131096,0.091637



Sample metrics for overall questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,"Description: The mark ""7-Eleven"" for a conveni...",17,4.352941,0.649493,15.668724,6.875000,0.0000
1,"Description: The mark ""Airbus"" for an airplane...",8,6.125000,0.585469,8.000000,11.130000,0.0000
2,"Description: The mark ""Amazon"" for an online s...",9,5.555556,0.488262,9.000000,8.897778,0.1779
3,"Description: The mark ""American Airlines"" for ...",11,6.090909,0.566378,11.000000,11.227273,0.0000
4,"Description: The mark ""Antilds"" for plant seeds.",7,5.428571,0.480939,7.000000,3.997143,0.0000



Sample metrics for top_f1 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Dominica. Do...,35,4.971429,0.585925,23.783034,11.463571,-0.6124
1,Question: Consider the country of Kyrgyzstan. ...,53,5.188679,0.669782,31.462799,16.341226,-0.3182
2,Question: Consider the country of Liechtenstei...,32,5.343750,0.592897,25.349928,12.037500,-0.3182
3,Question: Consider the country of Saint Lucia....,33,5.090909,0.552129,26.327314,11.584394,-0.3182
4,Question: Consider the country of Gabon. Does ...,54,4.444444,0.638847,31.086778,13.732593,0.8225



Sample metrics for top_f2 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Montenegro. ...,53,5.188679,0.669782,31.462799,16.563868,-0.3182
1,Question: Consider the country of Azerbaijan. ...,36,5.666667,0.556497,27.097109,14.374444,0.0000
2,Question: Consider the country of Sudan. Does ...,53,5.094340,0.669500,31.462799,16.118585,-0.3182
3,Question: Consider the country of Kazakhstan. ...,35,5.028571,0.594667,23.783034,11.126429,-0.6124
4,Question: Consider the country of Iran. Does t...,54,4.425926,0.641424,31.086778,13.732593,0.8225
